# MoCoP demo

Synthetic cohort of 3,000 patients with a known answer (no real patients).

* **Exposure** lowers C-reactive protein (LOINC `1988-5`) after the index date and lowers the risk of a
  secondary event (true risk ratio 0.60).
* Both raise the risk of the **outcome**.
* `truth.json` holds the true values, computed from both potential worlds of every simulated patient.

Run this notebook from the `examples/demo` folder after installing MoCoP (`pip install "git+https://github.com/clai-group/MoCoP.git"`).

In [ ]:
import json
import pandas as pd
from mocop import run

pd.set_option("display.width", 160)
truth = json.load(open("truth.json"))
truth

## 1. Input data

The four input files follow the schemas in the user guide (Section 3).

In [ ]:
pd.read_csv("cohort.csv").head()

In [ ]:
pd.read_csv("labs.csv").head()

In [ ]:
pd.read_csv("dx.csv").head()

In [ ]:
pd.read_csv("edges_curated.csv")

## 2. Run

All settings are in `config.yaml`. Outputs are written to `./output`.

In [ ]:
print(open("config.yaml").read())
res = run("config.yaml")

## 3. Trained edges

Estimated on the training split only, with balance and overlap checks.

In [ ]:
edges = res["edges_trained"].set_index("mediator")
edges[["kind", "n_treated", "estimate", "ci_low", "ci_high", "max_smd_after", "used"]].round(3)

In [ ]:
print(f"CRP severity shift: estimated {edges.loc['LAB_1988-5', 'estimate']:+.3f}, true {truth['crp_severity_shift_eligible']:+.3f}")
print(f"Event risk ratio:   estimated {edges.loc['EVENT', 'risk_ratio']:.3f}, true {truth['event_risk_ratio']:.3f}")

## 4. Outcome model

In [ ]:
res["model_performance"][["model", "split", "auc", "brier", "ece"]].round(3)

## 5. Effect on the outcome, by pathway

In [ ]:
eff = res["effects"]
eff[eff["group"] == "all patients"][["mode", "estimand", "risk_unexposed", "risk_exposed",
                                     "risk_ratio", "rr_ci_low", "rr_ci_high"]].round(3)

The estimated total effect is smaller than the true value (`truth["total_risk_ratio_all"]`) because MoCoP
represents post-index labs through their baseline values (see the user guide, Section 9). Direction,
pathway decomposition and edge estimates are recovered.

## 6. Checks

In [ ]:
res["directionality_audit"]